The code chunks below takes "tst_0001_pred_mask_2.png" from the folder "/Users/ruhisharmin/Library/CloudStorage/Box-Box/aether.lab/projects/Echo.Cardio/autosegmentation/Data/CAMUS/validation_pred_mask_2_Folder/patient0401". Then it 
1. it creates a binary mask from the pred_mask_2
2. from binary mask, creates a filteres image 
3. Convert filtered_mask to binary for 

Summary: Loading the image and creating a binary mask / Filtering small components using connected component analysis / Converting the filtered mask to binary format / Skeletonizing the image and visualize it

each processing step includes:
1. Original Image: Shows the input prediction mask as loaded from file
2. Binary Mask: Displays the image after thresholding (converting to pure black and white)
3. Filtered Mask: Shows only the large components after removing small noise elements
4. Skeletonized Mask: Displays the thinned skeleton of the filtered mask
5. Original with Skeleton Overlay: Original image with the skeleton contour plotted on top (in red)
6. Filtered Mask with Skeleton Overlay: Filtered mask with the skeleton contour plotted on top (in red)

The visualizations are in a 2×3 grid for comparison between the different processing stages. added code to extract the contour coordinates from the skeleton and print basic information about those coordinates. see:
- How well the initial thresholding captures the structures in the image
- What gets removed during the filtering step
- How the skeletonization process simplifies the structure
- How the final skeleton contour relates to both the original and filtered masks

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize

# Define path to the input image
image_path = '/Users/ruhisharmin/Library/CloudStorage/Box-Box/aether.lab/projects/Echo.Cardio/autosegmentation/Data/CAMUS/validation_pred_mask_2_Folder/patient0401/tst_0001_pred_mask_2.png'

# 1. Load the image and create binary mask
orig_unet_pred_mask_2 = cv2.imread(image_path, 0)  # Read as grayscale

# Check if image was loaded successfully
if orig_unet_pred_mask_2 is None:
    raise ValueError(f"Error: Could not load image from {image_path}. Please check the file path.")

# 2. Convert to binary mask (threshold)
binary_mask = (orig_unet_pred_mask_2 > 127).astype(np.uint8) * 255

# 3. Remove small elements using connected component analysis
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)

# 4. Filter components by area
min_area = 500  # Adjust this threshold to separate main arch from thin segments
filtered_mask = np.zeros_like(binary_mask)
for i in range(1, num_labels):  # Skip background (0)
    if stats[i, cv2.CC_STAT_AREA] > min_area:
        filtered_mask[labels == i] = 255

# 5. Convert filtered_mask to binary for skeletonization (0s and 1s)
binary = (filtered_mask > 0).astype(np.uint8)

# 6. Skeletonize the mask
skeleton = skeletonize(binary).astype(np.uint8)

# 7. Multiply by 255 to get a standard binary image
skeleton_img = skeleton * 255

# 8. Get coordinates of the skeleton contour
contours, _ = cv2.findContours(skeleton_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
if contours:
    main_contour = max(contours, key=cv2.contourArea)
    coordinates = main_contour.reshape(-1, 2)
else:
    coordinates = np.array([])

# 9. Visualize all processing steps
plt.figure(figsize=(15, 10))

# Original mask
plt.subplot(2, 3, 1)
plt.imshow(orig_unet_pred_mask_2, cmap='gray')
plt.title('Original Pred Mask')
plt.axis('off')

# Binary mask
plt.subplot(2, 3, 2)
plt.imshow(binary_mask, cmap='gray')
plt.title('Binary Mask')
plt.axis('off')

# Filtered mask (large components only)
plt.subplot(2, 3, 3)
plt.imshow(filtered_mask, cmap='gray')
plt.title('Filtered Mask (Large Components)')
plt.axis('off')

# Skeletonized mask
plt.subplot(2, 3, 4)
plt.imshow(skeleton_img, cmap='gray')
plt.title('Skeletonized Mask')
plt.axis('off')

# Skeleton contour on original
plt.subplot(2, 3, 5)
plt.imshow(orig_unet_pred_mask_2, cmap='gray')
if len(coordinates) > 0:
    plt.plot(coordinates[:, 0], coordinates[:, 1], 'r-', linewidth=1)
plt.title('Original with Skeleton Overlay')
plt.axis('off')

# Skeleton contour on filtered mask
plt.subplot(2, 3, 6)
plt.imshow(filtered_mask, cmap='gray')
if len(coordinates) > 0:
    plt.plot(coordinates[:, 0], coordinates[:, 1], 'r-', linewidth=1)
plt.title('Filtered Mask with Skeleton Overlay')
plt.axis('off')

plt.tight_layout()
plt.show()

# Print information about the contour if needed
if len(coordinates) > 0:
    print(f"Number of points in skeleton contour: {len(coordinates)}")
    print("First 5 coordinates:")
    print(coordinates[:5])

code below correctly identify all three points on our cardiac curve and display them with proper labeling.
- point3 denotes the APex
- end1 and end2 are the MV1 and MV2 points: splitting the image into top and bottom regions and identifying the endpoints in the bottom left and right portions of the curve.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

def detect_and_visualize_cardiac_points(image_path):
    """
    Detects and visualizes the three key points on a cardiac contour.
    
    Parameters:
    -----------
    image_path : str
        Path to the image file
        
    Returns:
    --------
    tuple
        (point3, end1, end2) coordinates as tuples
    """
    # Load image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image from {image_path}")
    
    # Ensure binary image
    if img.max() > 1:
        _, binary = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
    else:
        binary = img * 255
    
    # Get all white pixels
    y_coords, x_coords = np.where(binary > 0)
    
    if len(x_coords) < 3:
        raise ValueError("Not enough points in the image")
    
    # STEP 1: Find the top point (minimum y-coordinate) using specified logic
    min_y_idx = np.argmin(y_coords)
    point3 = (int(x_coords[min_y_idx]), int(y_coords[min_y_idx]))  # Topmost point
    
    # STEP 2: Find both endpoints
    # Calculate median x to separate left from right
    x_median = np.median(x_coords)
    
    # Find points in the bottom region (bottom 30%)
    y_max = np.max(y_coords)
    y_min = np.min(y_coords)
    bottom_threshold = y_min + 0.7 * (y_max - y_min)
    
    # Get bottom points
    bottom_mask = y_coords > bottom_threshold
    bottom_x = x_coords[bottom_mask]
    bottom_y = y_coords[bottom_mask]
    
    # If we don't find enough points in the bottom region, lower the threshold
    if len(bottom_x) < 2:
        bottom_threshold = y_min + 0.5 * (y_max - y_min)
        bottom_mask = y_coords > bottom_threshold
        bottom_x = x_coords[bottom_mask]
        bottom_y = y_coords[bottom_mask]
    
    # Split bottom points into left and right sides
    left_mask = bottom_x < x_median
    right_mask = bottom_x >= x_median
    
    # Find the bottommost point on the left side (MV anterior)
    if np.any(left_mask):
        left_y = bottom_y[left_mask]
        left_x = bottom_x[left_mask]
        left_bottom_idx = np.argmax(left_y)
        end1 = (int(left_x[left_bottom_idx]), int(left_y[left_bottom_idx]))
    else:
        # Fallback: use leftmost point
        left_idx = np.argmin(x_coords)
        end1 = (int(x_coords[left_idx]), int(y_coords[left_idx]))
    
    # Find the bottommost point on the right side
    if np.any(right_mask):
        right_y = bottom_y[right_mask]
        right_x = bottom_x[right_mask]
        right_bottom_idx = np.argmax(right_y)
        end2 = (int(right_x[right_bottom_idx]), int(right_y[right_bottom_idx]))
    else:
        # Fallback: use rightmost point
        right_idx = np.argmax(x_coords)
        end2 = (int(x_coords[right_idx]), int(y_coords[right_idx]))
    
    # Create visualization
    # Convert to RGB
    vis_img = np.zeros((binary.shape[0], binary.shape[1], 3), dtype=np.uint8)
    
    # Draw the curve in white
    vis_img[binary > 0] = [255, 255, 255]
    
    # Draw the three points with distinct colors and sizes
    marker_size = 8
    
    # Top point (Yellow)
    cv2.circle(vis_img, point3, marker_size, (0, 255, 255), -1)
    
    # End1 (Blue)
    cv2.circle(vis_img, end1, marker_size, (255, 0, 0), -1)
    
    # End2 (Red)
    cv2.circle(vis_img, end2, marker_size, (0, 0, 255), -1)
    
    # Add labels with outlines for better visibility
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    thickness = 2
    
    # Add text with black outline for better visibility
    def add_text_with_outline(img, text, position, color):
        cv2.putText(img, text, position, font, font_scale, (0, 0, 0), thickness + 2)
        cv2.putText(img, text, position, font, font_scale, color, thickness)
    
    add_text_with_outline(vis_img, "Top", (point3[0] + 10, point3[1]), (0, 255, 255))
    add_text_with_outline(vis_img, "End1", (end1[0] + 10, end1[1]), (255, 0, 0))
    add_text_with_outline(vis_img, "End2", (end2[0] + 10, end2[1]), (0, 0, 255))
    
    # Display the result
    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
    plt.title("Cardiac Curve Points")
    plt.axis('off')
    plt.show()
    
    # Print coordinates to console
    print(f"Top point: {point3}")
    print(f"End1 (left endpoint): {end1}")
    print(f"End2 (right endpoint): {end2}")
    
    return point3, end1, end2

# Example usage:
# point3, end1, end2 = detect_and_visualize_cardiac_points("cardiac_image.png")

This code below:

Extracts the skeleton centerline from the filtered mask
Identifies three corner points: the leftmost, rightmost, and topmost points of the contour
Calculates the circumcenter of these three points to use as the origin for polar coordinates
Falls back to the centroid if the circumcenter calculation fails
Converts the (X,Y) Cartesian coordinates to (r,θ) polar coordinates using this origin
Saves both coordinate systems to a CSV file in Google Drive
Creates a visualization showing:

The original contour
The circumcenter used as the origin
The three corner points used to calculate the circumcenter
The triangle formed by these three points
Sample radius lines to illustrate the polar coordinate system


The polar coordinates will give us r (distance from the circumcenter) and θ (angle in degrees, 0-360°) for each point along the contour.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize
from google.colab import drive ##  change
from datetime import datetime

# Mount Google Drive or local folder or Box: change accordingly
drive.mount('/content/gdrive')
loaddir_data = "/content/gdrive/My Drive/UNebraska"

# Make sure the directory exists
import os
os.makedirs(loaddir_data, exist_ok=True)

# Assuming filtered_mask is our first image
binary = (filtered_mask > 0).astype(np.uint8)

# Skeletonize the mask
skeleton = skeletonize(binary).astype(np.uint8)
skeleton_img = skeleton * 255

# Find contours in the skeleton
contours, _ = cv2.findContours(skeleton_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

if contours:
    # Get the largest contour
    main_contour = max(contours, key=cv2.contourArea)
    coordinates = main_contour.reshape(-1, 2)
    
    # Extract X and Y coordinates
    x_coords = coordinates[:, 0]
    y_coords = coordinates[:, 1]
    
    # Identify three corner points from the contour
    min_x_idx = np.argmin(x_coords)
    max_x_idx = np.argmax(x_coords)
    min_y_idx = np.argmin(y_coords)

    point1 = (x_coords[min_x_idx], y_coords[min_x_idx])  # Leftmost point
    point2 = (x_coords[max_x_idx], y_coords[max_x_idx])  # Rightmost point
    point3 = (x_coords[min_y_idx], y_coords[min_y_idx])  # Topmost point
    
    # Function to calculate circumcenter of three points
    def calculate_circumcenter(p1, p2, p3):
        # Convert points to numpy arrays for easier calculation
        p1 = np.array(p1)
        p2 = np.array(p2)
        p3 = np.array(p3)
        
        # Calculate midpoints of two sides
        mid1 = (p1 + p2) / 2
        mid2 = (p2 + p3) / 2
        
        # Calculate slopes of perpendicular bisectors
        # Handle infinite slope cases
        if p2[0] == p1[0]:  # Vertical line
            slope1 = 0  # Horizontal perpendicular
        else:
            slope1 = -1 / ((p2[1] - p1[1]) / (p2[0] - p1[0]))
            
        if p3[0] == p2[0]:  # Vertical line
            slope2 = 0  # Horizontal perpendicular
        else:
            slope2 = -1 / ((p3[1] - p2[1]) / (p3[0] - p2[0]))
        
        # Calculate y-intercepts
        b1 = mid1[1] - slope1 * mid1[0]
        b2 = mid2[1] - slope2 * mid2[0]
        
        # Handle special cases for vertical lines
        if abs(slope1 - slope2) < 1e-10:  # Nearly parallel
            # Fall back to centroid in case of nearly parallel lines
            return (p1 + p2 + p3) / 3
        
        # Calculate x-coordinate of intersection
        if p2[0] == p1[0]:  # First perpendicular is horizontal
            cx = mid1[0]
        elif p3[0] == p2[0]:  # Second perpendicular is horizontal
            cx = mid2[0]
        else:
            cx = (b2 - b1) / (slope1 - slope2)
        
        # Calculate y-coordinate of intersection
        if p2[0] == p1[0]:  # First perpendicular is horizontal
            cy = slope2 * cx + b2
        else:
            cy = slope1 * cx + b1
        
        return (cx, cy)

    # Calculate the circumcenter
    try:
        cx, cy = calculate_circumcenter(point1, point2, point3)
        
        # Ensure the result is finite
        if not (np.isfinite(cx) and np.isfinite(cy)):
            # Fall back to centroid if circumcenter calculation fails
            M = cv2.moments(main_contour)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
            else:
                cx = np.mean(x_coords).astype(int)
                cy = np.mean(y_coords).astype(int)
            print("Falling back to centroid due to circumcenter calculation issues.")
        else:
            print(f"Using circumcenter of three corner points as origin: ({cx:.1f}, {cy:.1f})")
    except Exception as e:
        # Fall back to centroid if there's any error
        M = cv2.moments(main_contour)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
        else:
            cx = np.mean(x_coords).astype(int)
            cy = np.mean(y_coords).astype(int)
        print(f"Error calculating circumcenter: {e}. Using centroid instead.")
    
    # Convert to polar coordinates
    # Shift coordinates to use the calculated center as origin
    x_centered = x_coords - cx
    y_centered = y_coords - cy
    
    # Calculate r and theta
    # Note: in image coordinates, y increases downward, so adjust for correct angles
    r = np.sqrt(x_centered**2 + y_centered**2)
    theta = np.arctan2(-y_centered, x_centered)  # Negative y for correct orientation
    
    # Convert theta from radians to degrees
    theta_deg = np.degrees(theta)
    
    # Make all angles positive (0 to 360 degrees)
    theta_deg = np.mod(theta_deg, 360)
    
    # Create array with both Cartesian and polar coordinates
    all_coordinates = np.column_stack((x_coords, y_coords, r, theta_deg))
    
    # Print the first 10 points in both coordinate systems
    print("First 10 points (x, y, r, θ):")
    for i in range(min(10, len(all_coordinates))):
        print(f"Point {i}: Cartesian ({all_coordinates[i,0]:.1f}, {all_coordinates[i,1]:.1f}), " + 
              f"Polar ({all_coordinates[i,2]:.1f}, {all_coordinates[i,3]:.1f}°)")
    
    # Save all coordinates to CSV
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f"{loaddir_data}/coordinates_cartesian_and_polar_circumcenter_{timestamp}.csv"
    np.savetxt(output_file, all_coordinates, delimiter=',', 
              header='x,y,r,theta_degrees', comments='')
    print(f"Coordinates saved to '{output_file}'")
    
    # Visualize the skeleton with circumcenter and the three corner points
    plt.figure(figsize=(12, 10))
    plt.imshow(filtered_mask, cmap='gray')
    plt.plot(x_coords, y_coords, 'r-', linewidth=2, label='Centerline')
    plt.plot(cx, cy, 'bo', markersize=8, label='Circumcenter (origin)')
    
    # Plot the three corner points used for circumcenter calculation
    plt.plot(point1[0], point1[1], 'go', markersize=6, label='Corner Point 1 (leftmost)')
    plt.plot(point2[0], point2[1], 'mo', markersize=6, label='Corner Point 2 (rightmost)')
    plt.plot(point3[0], point3[1], 'yo', markersize=6, label='Corner Point 3 (topmost)')
    
    # Draw lines connecting the three corner points to form a triangle
    plt.plot([point1[0], point2[0]], [point1[1], point2[1]], 'g--', alpha=0.5)
    plt.plot([point2[0], point3[0]], [point2[1], point3[1]], 'g--', alpha=0.5)
    plt.plot([point3[0], point1[0]], [point3[1], point1[1]], 'g--', alpha=0.5)
    
    # Optionally show a few radius lines to visualize the polar coordinates
    for angle in [0, 45, 90, 135, 180, 225, 270, 315]:
        rad = np.radians(angle)
        end_x = cx + 50 * np.cos(rad)
        end_y = cy - 50 * np.sin(rad)  # Negative because of image coordinates
        plt.plot([cx, end_x], [cy, end_y], 'c--', alpha=0.5)
    
    plt.title('Skeleton Centerline with Circumcenter as Polar Origin')
    plt.legend()
    plt.axis('off')
    plt.show()
else:
    print("No contours found in the skeleton image.")


This code:

Finds the contour points from the skeleton image
Identifies the three corner points (leftmost, rightmost, and topmost)
Creates a visualization showing:

The original binary mask as the background
The contour drawn as a red line
The three corner points marked with different colored circles
Annotations showing the exact coordinates of each point


Prints the coordinates of the three points to the console

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Assuming filtered_mask is binary mask image
# Find contours in the skeleton image (from previous code)
contours, _ = cv2.findContours(skeleton_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

if contours:
    # Get the largest contour
    main_contour = max(contours, key=cv2.contourArea)
    coordinates = main_contour.reshape(-1, 2)
    
    # Extract X and Y coordinates
    x_coords = coordinates[:, 0]
    y_coords = coordinates[:, 1]
    
    # Identify the three corner points from the contour
    min_x_idx = np.argmin(x_coords)
    max_x_idx = np.argmax(x_coords)
    min_y_idx = np.argmin(y_coords)

    point1 = (x_coords[min_x_idx], y_coords[min_x_idx])  # Leftmost point
    point2 = (x_coords[max_x_idx], y_coords[max_x_idx])  # Rightmost point
    point3 = (x_coords[min_y_idx], y_coords[min_y_idx])  # Topmost point
    
    # Create visualization
    plt.figure(figsize=(12, 10))
    plt.imshow(filtered_mask, cmap='gray')
    
    # Plot the contour
    plt.plot(x_coords, y_coords, 'r-', linewidth=2, label='Contour')
    
    # Plot the three corner points with different colors and larger markers
    plt.plot(point1[0], point1[1], 'go', markersize=10, label='Leftmost Point')
    plt.plot(point2[0], point2[1], 'bo', markersize=10, label='Rightmost Point')
    plt.plot(point3[0], point3[1], 'mo', markersize=10, label='Topmost Point')
    
    # Add annotations with coordinates
    plt.annotate(f"({point1[0]:.1f}, {point1[1]:.1f})", 
                 (point1[0]+5, point1[1]), color='green', fontsize=12)
    plt.annotate(f"({point2[0]:.1f}, {point2[1]:.1f})", 
                 (point2[0]-40, point2[1]), color='blue', fontsize=12)
    plt.annotate(f"({point3[0]:.1f}, {point3[1]:.1f})", 
                 (point3[0]+5, point3[1]-10), color='magenta', fontsize=12)
    
    plt.title('Three Corner Points on Contour')
    plt.legend(loc='upper left')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Print the coordinates
    print(f"Leftmost point (green): ({point1[0]:.1f}, {point1[1]:.1f})")
    print(f"Rightmost point (blue): ({point2[0]:.1f}, {point2[1]:.1f})")
    print(f"Topmost point (magenta): ({point3[0]:.1f}, {point3[1]:.1f})")
else:
    print("No contours found in the skeleton image.")